# Binary classification with an epoch-based trainer

`EpochTrainer` is the counterpart to `SimpleTrainer` (see `simpletrainer_examples.ipynb`) for torch models. Instead of one non-resumable `.fit()` call, it trains epoch by epoch, so there is a per-epoch loss to inspect, checkpointing, and a trainer that can be resumed from a saved snapshot.

It uses the same binary AGN-vs-star-forming data as the torch section of `simpletrainer_examples.ipynb`, so the two notebooks are easy to compare. Calibration is not included here yet - `EpochTrainer` has a `calibrator_type` slot reserved for it, but ANN calibration works differently from sklearn's `CalibratedClassifierCV` and is covered separately.

## Data Preprocessing 
As discussed in the `simpletrainer_examples` notebook, we are using a medal system for data quality. Here, we preprocess data from `bronze` (raw data) to `silver` (preprocessed and ready to be made into actual training datasets). This is a repeptition of the respective cell in the `simpletrainer_examples` notebook.

In [ ]:
from pathlib import Path
import pandas as pd

input_path = Path("../data/bronze/default")
outpath = Path("../data/silver/default")  # note the output to the 'silver' data level
outpath.mkdir(exist_ok=True, parents=True)

for v in Path(input_path).glob("*.dat"):
    print("file: ", v)
    source = None
    if "AGN" in v.name:
        source = 0
    elif "POPSTAR" in v.name:
        source = 1
    else:
        raise ValueError(f"Unknown source for {v.name}")

    df = pd.read_csv(
        v,
        sep=r"\s+",
        engine="python",
        comment="#",
        na_values=["nan", "NaN"],
    )

    df["source"] = source
    df.to_csv(outpath / f"{v.stem}.csv", index=False, sep=",")
    print(df.head(3))

## Split silver data into gold data train/val/test

`SimpleTrainer`'s notebook builds one dataset and slices it with `random_split` at call time, because `trainer.fit(train_dataset)` and `trainer.evaluate(test_dataset)` take the split as an argument. `EpochTrainer` instead builds its own train, validation and test datasets from config when it is constructed, and `train()`/`evaluate()` take no dataset argument - so the three splits need to already exist on disk as separate directories before the trainer is built. How to split and compose datasets is often task- and training runs specific. This split data will be used directly for training ML models, and hence constitutes the `gold` level. We consider it trustworthy and clean. 

The cell below reads `../data/silver/default` (the same AGN=0/POPSTAR=1 grid used in the torch section of `simpletrainer_examples.ipynb`), and writes a 70/15/15 stratified split to `../data/gold/epoch_trainer_example/{train,val,test}`.

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

inpath = Path("../data/silver/default")
outpath = Path("../data/gold/epoch_trainer_example")

data = pd.concat(
    [pd.read_csv(f) for f in sorted(inpath.glob("*.csv"))], ignore_index=True
)

train_df, rest_df = train_test_split(
    data, test_size=0.3, stratify=data["source"], random_state=42
)
val_df, test_df = train_test_split(
    rest_df, test_size=0.5, stratify=rest_df["source"], random_state=42
)

for name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    split_dir = outpath / name
    split_dir.mkdir(parents=True, exist_ok=True)
    split_df.to_csv(split_dir / "data.csv", index=False)
    print(name, split_df.shape, split_df["source"].value_counts().to_dict())

## Load the config

Like `SimpleTrainer`'s config, this is a plain YAML file with dotted import paths for the model, loss, optimizer and metrics. The difference is that there is no separate `dataset:` block: `train_dataset_type/args/kwargs`, `val_dataset_type/args/kwargs` and `test_dataset_type/args/kwargs` are just more constructor arguments, pointing at the three split directories written above.

In [ ]:
from pprint import pprint
import yaml

with open("../configs/binary_classifier_epoch_example.yaml", "r") as f:
    config = yaml.safe_load(f)

pprint(config["trainer"])

## Transform

The config's `transform: __main__.to_float32` names a function that has to exist in the notebook's own kernel namespace (`__main__`) by the time the trainer is built - the same way `simpletrainer_examples.ipynb` uses `__main__.transform_types` for its torch config. Hugging Face Datasets passes the function a batch dictionary and expects a dictionary back. The feature columns are converted to float32 tensors by `TabularDataset`; this transform casts the label values to floats, which is what `BCEWithLogitsLoss` expects.

In [ ]:
def to_float32(batch: dict) -> dict:
    batch = dict(batch)
    batch["source"] = [float(value) for value in batch["source"]]
    return batch

## Build the trainer

`EpochTrainer.from_config` builds the train/val/test datasets, the model (`torchvision.ops.MLP`: `Linear(18, 64) -> ReLU -> Linear(64, 1)`, wrapped in skorch's `NeuralNetBinaryClassifier` because `task: binary-classification`), the optimizer, the loss, and the callbacks (checkpointing, progress bar, and one `EpochScoring` callback per configured metric) all in one call.

In [ ]:
from GalaxySpectrumClassifier import EpochTrainer
import yaml

with open("../configs/binary_classifier_epoch_example.yaml", "r") as f:
    config = yaml.safe_load(f)
trainer = EpochTrainer.from_config(config["trainer"])

## Train

`train()` runs the configured `max_epochs` passes over the training dataset, validating against the validation dataset after every epoch. Unlike `SimpleTrainer.fit()`, this is resumable: calling `train()` again only trains the epochs still missing up to `max_epochs`, which is what makes `load_snapshot` + `train()` a real "continue training" further down.

In [ ]:
trainer.train()

## Inspect the per-epoch history

We can inspect the recorded metrics per epoch with the model's `history` attribute. Here, we are interested in the loss over epochs on training and validation datasets

In [ ]:
import matplotlib.pyplot as plt

train_loss = trainer.model.history[:, "train_loss"]
valid_loss = trainer.model.history[:, "valid_loss"]

plt.plot(train_loss, label="train_loss", marker="o")
plt.plot(valid_loss, label="valid_loss", marker="o")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()
plt.show()

# Test model

`evaluate()` scores the trainer's test dataset with every configured metric and returns a plain `{name: score}` dict, the same shape `SimpleTrainer.evaluate()` returns. As with the simple-trainer notebook, these scores come out pathologically high - that is the AGN/POPSTAR grids being easy to separate, not a property of the trainer.

In [ ]:
import pandas as pd

test_results = trainer.evaluate()
test_results = pd.DataFrame.from_dict(data=[test_results])
test_results.to_csv(trainer.output_path / "test_results.csv", index=False)
test_results

## Save and resume from a snapshot

`save_snapshot` writes `config.yaml` plus the model, optimizer, criterion and epoch history under the trainer's `output_path`. `load_snapshot` rebuilds the trainer from that config and restores all four, so training can continue exactly where it left off - this is the resumable state `SimpleTrainer` does not have. By default the restored trainer reuses the saved output directory; pass `save_to=...` to direct subsequent checkpoints, snapshots and exports to a new base directory.

In [ ]:
trainer.save_snapshot("trained_mlp")

loaded_trainer = EpochTrainer.load_snapshot(trainer.output_path / "trained_mlp")

# Same test dataset, same restored model, so this reproduces the scores above.
loaded_trainer.evaluate()

## Export for deployment

`export_model` writes just the trained weights and a small manifest, without the resumable optimizer/history state a snapshot carries. `export_format: default` here is skorch's own parameter format; `pt` is also available.

In [ ]:
trainer.export_model("exported_mlp")
sorted(p.name for p in (trainer.output_path / "exported_mlp").iterdir())

## Build custom model 

We can define our own pytorch model class and train that. Here, we will go with a 1D convolutional network for demonstration purposes. This is also available via torchvision. 

The following cells demonstrates the principle by building a general sequential feedforward network first, then building a conv1d based one using it 

In [ ]:
import torch
from GalaxySpectrumClassifier.utils import load_type
from typing import Any


class GeneralSequentialModel(torch.nn.Module):
    def __init__(self, layer_defs: list[dict[str, Any]]):
        super(GeneralSequentialModel, self).__init__()

        layers = []
        for lkw in layer_defs:
            type = load_type(lkw["type"])
            args = lkw["args"]
            kwargs = lkw["kwargs"]
            instance = type(*args, **kwargs)
            layers.append(instance)

        self.layers = torch.nn.Sequential(
            *layers
        )  # takes care of sequential application fo layers

    def forward(self, input):
        return self.layers(input)

we define the content of the network, layer by layer. This goes into the config later: 

In [ ]:
layer_def = [
    {
        # input conv1d layer
        "type": "torch.nn.Conv1d",
        "args": [
            18,
            64,
            4,
        ],  # all columns input (18), 64 dims output -> kernel size = 4 (very small)
        "kwargs": {"stride": 2, "padding": 0, "bias": True},
    },
    {
        # nonlinearity/activation
        "type": "torch.nn.ReLU",
        "args": [],
        "kwargs": {
            "inplace": False  # have it stateless -> pure function
        },
    },
    {
        # output conv1d layer
        "type": "torch.nn.Conv1d",
        "args": [64, 1, 4],
        "kwargs": {"stride": 2, "padding": 0, "bias": True},
    },
]

network = GeneralSequentialModel(layer_def)


# dummy usage of network:
t = torch.rand(18, 16, dtype=torch.float32)

network.forward(t)

# Train model using custom model 
We can use this model in the config as demonstrated by `binary_classifier_epoch_custom_model_example.yaml`.
Here, we are using a convolutional neural network. For our data, this does not make much sense, because the individual columns are not part of a geometrically coherent structure like a sequence or grid. We use it here for demonstration purposes only.

In [ ]:
from GalaxySpectrumClassifier import EpochTrainer
import yaml
from typing import Any
import torch
from pprint import pprint


# needed for type conversion
def convert_to_float32(batch: dict) -> dict:
    batch = dict(batch)
    batch["source"] = [float(value) for value in batch["source"]]
    return batch


class ConvModel(torch.nn.Module):
    def __init__(self, layer_defs: list[dict[str, Any]]):
        super().__init__()

        layers = []
        for lkw in layer_defs:
            type = load_type(lkw["type"])
            args = lkw["args"]
            kwargs = lkw["kwargs"]
            instance = type(*args, **kwargs)
            layers.append(instance)

        self.layers = torch.nn.Sequential(
            *layers
        )  # takes care of sequential application fo layers

    def forward(self, input):
        # the batch from the dataset needs adjustment for conv1d models
        input = input.unsqueeze(1)  # (B, 18) -> (B, 1, 18)
        output = self.layers(input)
        return output.flatten(start_dim=1).squeeze(-1)


with open("../configs/binary_classifier_epoch_custom_model_example.yaml", "r") as f:
    config = yaml.safe_load(f)

trainer = EpochTrainer.from_config(config["trainer"])

pprint(trainer.config)

We can then proceed to train the model in the usual way: 

In [ ]:
trainer.train()
results = trainer.evaluate()
results

Convolutional neural networks are built for topologically structured data - grids, sequences, and with modifications even general graphs (graph convolutional networks), but we don't have this structure in our case. Consequently, the inductive bias of the model is not fitting the data, and results are comparatively poor. 

We can export the mdoel again if we want:

In [ ]:
trainer.export_model("pytorch_export_conv")

## A note on cross-validation

`simpletrainer_examples.ipynb` ends with a `StratifiedKFold` loop, refitting the same trainer on each fold's subset. `EpochTrainer` builds its train/val/test datasets once, at construction, and `train()`/`evaluate()` don't take a dataset argument - so each fold would need its own freshly-built `EpochTrainer` and its own `output_path`. That's left out of this notebook for now to keep it short and simple. While the principle is the same as before, convenience methods in `EpochTrainer` to do that are currently not implemented, and the dataset-model plumbing would need to be done by hand.